# RetinexFormer v1 — Epoch training with warm-up

Proposed Canny-guided model only. **200 epochs total, including 3 warm-up epochs**.
LR: `1e-6 → 1e-4 → 1e-6`. Training pairs are split 90%/10% with seed 42;
the official test set is used only after training to evaluate the best validation checkpoint.

Run setup, Drive mount, configuration, data, and model cells in order. Set a new
experiment name for a fresh run, or `RESUME_FROM` to that run's `checkpoints/last.pth`.
The training cell starts the full run; optional tests run separately without modifying its model.
This notebook requires the epoch-training changes to be pushed to GitHub before remote setup.


In [ ]:
# Configure deterministic CUDA math before any CUDA work.
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
%cd /content
!if [ ! -d /content/myRetinexformer/.git ]; then git clone https://github.com/Silly-Gooses/myRetinexformer.git /content/myRetinexformer; fi
import subprocess
for command in (
    ["git", "fetch", "origin", "v1"],
    ["git", "switch", "v1"],
    ["git", "pull", "--ff-only", "origin", "v1"],
):
    subprocess.run(command, cwd="/content/myRetinexformer", check=True)
%cd /content/myRetinexformer
from pathlib import Path
assert Path("training_utils.py").is_file(), "Push the epoch-training update to v1 before running this notebook."
print("Version 1 epoch-training code is available")


In [ ]:
!pip install -q einops opencv-python pillow matplotlib


## Google Drive and experiment configuration


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import json
from canny import CannyEdgeDetector
from training_utils import seed_everything

DATASET_ROOT = Path("/content/drive/MyDrive/datasets/LOLv2/Real_captured")
EXPERIMENT_NAME = "Lightweight_RetinexFormer_v1_LOLv2_real_Epoch200"
FULL_ROOT = Path("/content/drive/MyDrive/RetinexFormer_full_training") / EXPERIMENT_NAME
CKPT_DIR = FULL_ROOT / "checkpoints"
LOG_DIR = FULL_ROOT / "logs"
VIS_DIR = FULL_ROOT / "visuals"

NUM_EPOCHS = 200
WARMUP_EPOCHS = 3
WARMUP_START_LR = 1e-6
PEAK_LR = 1e-4
MIN_LR = 1e-6
SEED = 42
VAL_FRACTION = 0.10
BATCH_SIZE = 8
PATCH_SIZE = 256
NUM_WORKERS = 2
USE_AMP = False
VIS_EVERY = 10
FIXED_VIS_COUNT = 5

# Fresh run: None. Resume: CKPT_DIR / "last.pth".
# Keep the original NUM_EPOCHS and all other training settings when resuming.
RESUME_FROM = None

CONFIG = {
    "version": "v1", "training_mode": "epoch", "seed": SEED,
    "dataset_root": str(DATASET_ROOT), "val_fraction": VAL_FRACTION,
    "num_epochs": NUM_EPOCHS, "warmup_epochs": WARMUP_EPOCHS,
    "warmup_start_lr": WARMUP_START_LR, "peak_lr": PEAK_LR, "min_lr": MIN_LR,
    "batch_size": BATCH_SIZE, "patch_size": PATCH_SIZE, "num_workers": NUM_WORKERS,
    "use_amp": USE_AMP, "vis_every": VIS_EVERY, "fixed_vis_count": FIXED_VIS_COUNT,
    "optimizer": {"name": "Adam", "betas": [0.9, 0.999], "weight_decay": 0.0},
    "loss": "L1", "edge_guidance": True,
    "architecture": {"in_channels": 3, "out_channels": 3, "n_feat": 40,
                     "stage": 1, "num_blocks": [1, 2, 2]},
    "canny": CannyEdgeDetector().config,
    "deterministic": True,
}
CONFIG = json.loads(json.dumps(CONFIG))
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if USE_AMP and device.type != "cuda":
    raise ValueError("USE_AMP=True requires CUDA in this notebook")
print("Device:", device)
print("Output root:", FULL_ROOT)


## Fixed training / validation split

The manifest records exact filenames. Training uses random crops and geometric
augmentation; validation uses full images without augmentation. The final partial
training batch is included. Resume checks membership and configuration before loading.


In [ ]:
from analysis_utils import find_paired_images
from training_utils import prepare_experiment, build_epoch_loaders

train_low = DATASET_ROOT / "Train/Low"
train_normal = DATASET_ROOT / "Train/Normal"
pairs = find_paired_images(train_low, train_normal)
split = prepare_experiment(FULL_ROOT, CONFIG, [name for name, _, _ in pairs], RESUME_FROM)
train_loader_full, val_loader_full, loader_generators = build_epoch_loaders(
    train_low, train_normal, split, batch_size=BATCH_SIZE, patch_size=PATCH_SIZE,
    num_workers=NUM_WORKERS, seed=SEED, pin_memory=(device.type == "cuda"),
)
print("Training pairs:", len(train_loader_full.dataset))
print("Validation pairs:", len(val_loader_full.dataset))
print("Batches per epoch:", len(train_loader_full))
print("Split manifest:", FULL_ROOT / "split.json")


## Fresh model and epoch schedule

This creates a fresh v1 model, not a smoke-test model. Resume restores all training
state inside the training call. Warm-up LR is `1e-6`, `5.05e-5`, `1e-4` in epochs
1–3. Epochs 4–200 use cosine progress `(epoch - 3) / 197`, ending at `1e-6`.
The LR is constant within each epoch.


In [ ]:
from RetinexFormer_arch import RetinexFormer
from training_utils import EpochWarmupCosine

# Re-running model setup starts from the same seed; resume restores saved RNG later.
seed_everything(SEED)
model_full = RetinexFormer(**CONFIG["architecture"], edge_guidance=True,
                           canny_config=CONFIG["canny"]).to(device)
optimizer_full = torch.optim.Adam(model_full.parameters(), lr=WARMUP_START_LR,
                                  betas=tuple(CONFIG["optimizer"]["betas"]), weight_decay=0.0)
scheduler_full = EpochWarmupCosine(
    optimizer_full, num_epochs=NUM_EPOCHS, warmup_epochs=WARMUP_EPOCHS,
    warmup_start_lr=WARMUP_START_LR, peak_lr=PEAK_LR, min_lr=MIN_LR,
)
scaler_full = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print("Trainable parameters:", sum(p.numel() for p in model_full.parameters()))
print("Resume from:", RESUME_FROM or "Fresh training")


## Optional correctness checks (no full training)

Runs synthetic tests in a separate process. CUDA tests skip on CPU. This does not
change `model_full`. Dataset-backed training begins only in the next section.


In [ ]:
!python -m unittest discover -s tests -v


## Run epoch training

Every epoch logs validation L1/PSNR/SSIM and updates `last.pth`. A strictly better
validation PSNR updates `best.pth`; ties keep the earlier best. Visuals use fixed
validation samples at epoch 1, every 10 epochs, and the final epoch.

If interrupted mid-epoch, restart from `last.pth`; the incomplete epoch is repeated.
Do not call this cell twice without setting resume and re-running setup.


In [ ]:
from training_utils import run_epoch_training

history = run_epoch_training(
    model_full, train_loader_full, val_loader_full, optimizer_full,
    scheduler_full, scaler_full, device, CONFIG, split, FULL_ROOT,
    loader_generators, resume_from=RESUME_FROM,
)
print("Training finished. Best:", CKPT_DIR / "best.pth")
print("Last:", CKPT_DIR / "last.pth")
print("CSV:", LOG_DIR / "training_metrics.csv")


## Training curves


In [ ]:
import csv
import matplotlib.pyplot as plt

with (LOG_DIR / "training_metrics.csv").open(newline="") as file:
    rows = list(csv.DictReader(file))
if not rows:
    raise ValueError("No completed epochs to plot yet")
epochs = [int(row["epoch"]) for row in rows]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for key in ("train_loss", "val_loss"):
    axes[0, 0].plot(epochs, [float(row[key]) for row in rows], label=key)
axes[0, 0].set_title("Mean L1 loss")
axes[0, 0].legend()
for axis, key, title in ((axes[0, 1], "val_psnr", "Validation PSNR"),
                         (axes[1, 0], "val_ssim", "Validation SSIM"),
                         (axes[1, 1], "lr", "Learning rate used")):
    axis.plot(epochs, [float(row[key]) for row in rows])
    axis.set_title(title)
axes[1, 1].set_yscale("log")
for axis in axes.flat:
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(LOG_DIR / "training_curves.png", dpi=150)
plt.show()


## Inspect validation edge guidance

Display a saved Low / Edge / Lit-up / Output / GT grid. The individual panels are
also saved in `low/`, `edge/`, `lit_up/`, `output/`, and `gt/` under each visual epoch.


In [ ]:
from IPython.display import display
from PIL import Image

visual_epochs = sorted(VIS_DIR.glob("epoch_*"))
if visual_epochs:
    grids = sorted(visual_epochs[-1].glob("*.png"))
    if grids:
        print(grids[0])
        display(Image.open(grids[0]))
else:
    print("No completed visual epoch yet")


## Final test evaluation — best validation model

Run after training completes. Only here is the official LOLv2 Real test set
loaded. All its images are evaluated; no test image participates in training or
best-checkpoint selection. Reports are for the proposed model only.


In [ ]:
from torch.utils.data import DataLoader
from dataLoader import paired_image_Dataset
from analysis_utils import export_test_set
from training_utils import load_epoch_checkpoint

last = load_epoch_checkpoint(CKPT_DIR / "last.pth", CONFIG, split)
if last["completed_epoch"] != NUM_EPOCHS:
    raise ValueError("Complete the configured training run before final test evaluation")
best = load_epoch_checkpoint(CKPT_DIR / "best.pth", CONFIG, split)
model_full.load_state_dict(best["model"], strict=True)
model_full.eval()
test_low, test_normal = DATASET_ROOT / "Test/Low", DATASET_ROOT / "Test/Normal"
find_paired_images(test_low, test_normal)
test_dataset = paired_image_Dataset(str(test_low), str(test_normal), split="test",
                                    crop_size=None, augment=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)
FINAL_TEST_DIR = FULL_ROOT / "final_test_outputs"
psnr, ssim = export_test_set(model_full, test_loader, device, FINAL_TEST_DIR)
print("Test images:", len(test_dataset))
print("Best epoch:", best["best"]["epoch"])
print(f"Test PSNR: {psnr:.4f} | SSIM: {ssim:.4f}")
print("Outputs and metrics:", FINAL_TEST_DIR)


## Resume after a Colab disconnect

1. Run setup and mount Drive.
2. Keep the same experiment name, dataset, model, seed, batch size and schedule.
3. Set `RESUME_FROM = CKPT_DIR / "last.pth"` in configuration.
4. Run split, model, and training cells. Continue at the next completed-epoch boundary.

Model, optimizer, scheduler, scaler, split, history and random states are restored.
CSV is rebuilt from checkpoint history to remove duplicate/stale rows. Warm-up
is not restarted. Old iteration checkpoints cannot resume this epoch workflow.
Reproducibility is tested within one environment; different hardware/PyTorch
versions can produce different floating-point results. Deterministic CUDA
settings require starting from the runtime-configuration cell on a fresh runtime.


## Optional GPU performance check
Includes warm-up and Canny convergence overhead; does not alter training checkpoints.


In [ ]:
!python tests/benchmark_canny_v1.py --device cuda --size 256 --batch-size 8 --steps 10
